# 2. Ablations and controls
What does the model use? Feature ablations, transposed and shuffled test pieces, interval representation.

Run `1_train.ipynb` first: the transposition and shuffling tests use the trained models of the proposed model (`full`).

In [ ]:
import torch

from constants import *
from utils import get_device
from experiments import run_experiment, evaluate_trained_model
from disk_utils import load_results
from result_tables import per_seed_table, summary_table, paired_tests

In [ ]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True makes a tiny dry run of every step (1 epoch, 3 batches)
# and writes to results_quick_test/. Use it to check the setup.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    num_epochs = 1
    max_train_batches = 3
else:
    results_dir = RESULTS_DIR
    num_epochs = NUM_EPOCHS
    max_train_batches = None

## Feature ablations
The model is trained again without some of the note attributes.

Without duration

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="no_dur",
        seed=seed,
        device=device,
        features=["pc", "acc", "oct", "meas"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

Without measure position

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="no_meas",
        seed=seed,
        device=device,
        features=["pc", "acc", "oct", "dur"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

Without duration and measure position

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="no_rhythm",
        seed=seed,
        device=device,
        features=["pc", "acc", "oct"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

Without octave

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="no_oct",
        seed=seed,
        device=device,
        features=["pc", "acc", "dur", "meas"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

Pitch class and accidental only

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="pc_acc",
        seed=seed,
        device=device,
        features=["pc", "acc"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

Pitch class only

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="pc_only",
        seed=seed,
        device=device,
        features=["pc"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

## Absolute pitch
The proposed model is tested on transposed test pieces (no training).

In [ ]:
for seed in SEEDS:
    for kommas in TRANSPOSITIONS:
        evaluate_trained_model(
            tag="full",
            seed=seed,
            device=device,
            test_transpose=kommas,
            results_dir=results_dir
        )

Interval representation: the model sees the komma distance to the previous note instead of the pitch, with and without duration and measure position.

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="interval",
        seed=seed,
        device=device,
        features=["int", "dur", "meas"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="interval_only",
        seed=seed,
        device=device,
        features=["int"],
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

Transposition augmentation: every training piece is transposed by a random number of kommas. The trained model is then also tested on transposed test pieces.

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="transpose_aug",
        seed=seed,
        device=device,
        train_transpose_aug=TRANSPOSE_AUG_RANGE,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

In [ ]:
for seed in SEEDS:
    for kommas in [4, 9]:
        evaluate_trained_model(
            tag="transpose_aug",
            seed=seed,
            device=device,
            test_transpose=kommas,
            results_dir=results_dir
        )

## Note order
The proposed model is tested on test pieces with shuffled note order (no training).

In [ ]:
for seed in SEEDS:
    evaluate_trained_model(
        tag="full",
        seed=seed,
        device=device,
        test_shuffle=True,
        results_dir=results_dir
    )

A model that is trained on shuffled pieces, tested on shuffled test pieces.

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="train_shuffle",
        seed=seed,
        device=device,
        train_shuffle=True,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

In [ ]:
for seed in SEEDS:
    evaluate_trained_model(
        tag="train_shuffle",
        seed=seed,
        device=device,
        test_shuffle=True,
        results_dir=results_dir
    )

## Results so far

In [ ]:
results = load_results(results_dir)
per_seed = per_seed_table(results)
summary = summary_table(per_seed)
summary[["tag", "perturbation", "n", "acc_mean", "acc_ci", "f1_mean", "f1_ci"]]

Paired comparison of every variant with the proposed model over the five splits: mean accuracy difference in percentage points and paired t-test.

In [ ]:
paired_tests(per_seed, reference_tag="full")